# 07 — Ground-Truth Validation (Step 5)

**Goal:** hold our weak-proxy findings against a **real labeled fake-review dataset** (Hollenbeck et al., MIT-licensed; 381,734 labeled Amazon reviews). We re-run the trust classifier on the real `is_fake` label through the *identical* pipeline, compare ROC/PR to the verified-purchase proxy, and ask which signals actually survive.

> **Weak-proxy caveat.** Our primary label elsewhere is Amazon *Verified Purchase* — purchase verification, **not** deception. This chapter is the reality check.

> **Label-circularity caveat.** The dataset's `primary` rule defines fake as a *5-star* review of a fake-review product by a fake reviewer — so `rating` is predictive *by construction*. Read rating's importance as definitional, not discovered; the `labeled` strategy (human reviewer labels, rating-independent) is a cleaner test of behavioral/linguistic signals.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
from src import utils, viz
config = utils.load_config()
utils.set_seeds(config["project"]["random_seed"])
viz.set_house_style()

## 1. Real-label classifier (ground truth)
Load → clean → run the same KNN/RF pipeline on `is_fake`. (Set `groundtruth.sample_rows` in `config.yaml`; reduce it if the feature pipeline is slow on your machine.)

In [ ]:
from src import groundtruth_loader as gtl, clean, models, features

gt = clean.clean_reviews(gtl.load_groundtruth(config))
print('rows:', len(gt), '| fake rate:', round(gt['is_fake'].mean(), 3))
real = models.run_classification(gt, config, label_col='is_fake')
print({k: round(v['roc_auc'], 3) for k, v in real['results'].items()})
viz.plot_roc_curves(real['results']); plt.show()
viz.plot_pr_curves(real['results']); plt.show()
viz.plot_feature_importances(real['results']['random_forest']['importances']); plt.show()

### Compare label strategies
`primary` embeds rating==5; `labeled` uses human reviewer labels (rating-independent). Compare how the story changes — this is the heart of the honesty argument.

In [ ]:
for strat in ['primary', 'primary_or_deleted', 'labeled']:
    g = clean.clean_reviews(gtl.load_groundtruth(config, strategy=strat, sample_rows=8000))
    res = models.run_classification(g, config, label_col='is_fake')
    aucs = {k: round(v['roc_auc'], 3) for k, v in res['results'].items()}
    print(f'{strat:18s} fake_rate={g["is_fake"].mean():.3f}  {aucs}')

## 2. Proxy-label classifier
Run the *same* pipeline on the verified-purchase proxy. Replace the illustrative synthetic data below with your collected dataset from **notebook 01** (`config['data_source']['source']`).

In [ ]:
import tempfile, pandas as pd
from src import fallback_loader

# --- Illustrative proxy data; swap in your real collected table from notebook 01. ---
tmp = tempfile.mkdtemp()
fallback_loader.make_synthetic_sample(tmp, categories=('Electronics', 'Books'),
                                      n_products=12, reviews_per_product=12)
rows = []
for c in ('Electronics', 'Books'):
    rows += fallback_loader.load_category(config, c, raw_dir=tmp)
proxy = clean.clean_reviews(pd.DataFrame(rows))
proxy_res = models.run_classification(proxy, config)  # label=verified_purchase proxy
print({k: round(v['roc_auc'], 3) for k, v in proxy_res['results'].items()})
viz.plot_roc_curves(proxy_res['results']); plt.show()

## 3. Which signals survive? (per-feature univariate AUC)
For each feature, compare standalone separability under the **real** label vs the **proxy**. High on both = a robust signal; high on proxy only = a proxy artifact; high on real only = something the proxy misses.

In [ ]:
real_X = models.design_matrix(features.build_feature_table(gt, config))
real_auc = models.univariate_feature_auc(real_X, gt['is_fake'])

pf = features.build_feature_table(proxy, config)
py, _ = models.make_label(pf)
proxy_auc = models.univariate_feature_auc(models.design_matrix(pf).loc[py.index], py)

import pandas as pd
comp = pd.DataFrame({'real_auc': real_auc, 'proxy_auc': proxy_auc}).fillna(0.5)
def verdict(r):
    hi = 0.60
    if r.real_auc >= hi and r.proxy_auc >= hi: return 'survives (both)'
    if r.proxy_auc >= hi: return 'proxy-only (misleading)'
    if r.real_auc >= hi: return 'real-only (proxy misses)'
    return 'weak'
comp['verdict'] = comp.apply(verdict, axis=1)
comp.sort_values('real_auc', ascending=False).round(3)

## 4. Blind spots — honest write-up
Summarize for the poster/talk:
- **Where the proxy agrees with truth** (signals that survive): …
- **Proxy artifacts** (predict verified-purchase but not real fakes): …
- **What the proxy misses** (real-fake signals invisible to the verified flag): …
- **The rating-circularity lesson:** a label partly defined by a feature inflates that feature's apparent power — exactly why ground-truth validation matters.
- **Bottom line:** the verified-purchase proxy is a useful but imperfect stand-in; state plainly where it would mislead a platform that trusted it.